# Neighborhood Weather Agent — ADK Callbacks (Simulated Model)

This notebook uses live NWS data, a deterministic simulated ADK model, and `before_model_callback` / `after_model_callback`. No Gemini or Vertex AI credentials required. **Simulated responses are not generative AI.** Run cells top to bottom. The agent supports five US cities only.

In [1]:
from google.adk.agents import Agent


In [2]:
from google.adk.models.lite_llm import LiteLlm

In [3]:
%pip install -q -U google-adk requests

## 1. Live National Weather Service tool

In [4]:
import requests

CITIES = {
    "denver": (39.7392, -104.9903),
    "washington": (38.9072, -77.0369),
    "chantilly": (38.8943, -77.4311),
    "manhattan": (40.7831, -73.9712),
    "los angeles": (34.0522, -118.2437),
}

# Replace with a real contact address for regular NWS API use.
HEADERS = {"User-Agent": "NeighborhoodWeatherAgent/1.0 (contact: student@example.com)",
           "Accept": "application/geo+json"}

def get_weather(city: str) -> dict:
    """Get NWS temperature, precipitation probability, and humidity for a supported US city."""
    key = city.strip().lower()
    if key not in CITIES:
        return {"status": "error", "message": "Unsupported location. US cities only: " + ", ".join(CITIES)}
    lat, lon = CITIES[key]
    try:
        session = requests.Session()
        session.headers.update(HEADERS)
        def fetch(url):
            response = session.get(url, timeout=15)
            response.raise_for_status()
            return response.json()
        point = fetch(f"https://api.weather.gov/points/{lat},{lon}")["properties"]
        stations = fetch(point["observationStations"]).get("features", [])
        observation = None
        for station in stations[:8]:
            sid = station.get("properties", {}).get("stationIdentifier")
            if not sid:
                continue
            try:
                props = fetch(f"https://api.weather.gov/stations/{sid}/observations/latest")["properties"]
                if props.get("temperature", {}).get("value") is not None:
                    observation = props
                    break
            except (requests.RequestException, KeyError, ValueError, TypeError):
                continue
        if observation is None:
            raise ValueError("No recent temperature observation available")
        celsius = observation["temperature"]["value"]
        humidity = (observation.get("relativeHumidity") or {}).get("value")
        hourly = fetch(point["forecastHourly"]).get("properties", {}).get("periods", [])
        pop = (hourly[0].get("probabilityOfPrecipitation") or {}).get("value") if hourly else None
        return {
            "status": "success", "city": key.title(),
            "temperature_f": round(celsius * 9 / 5 + 32, 1),
            "humidity_percent": round(humidity, 1) if humidity is not None else None,
            "precipitation_probability_percent": pop,
        }
    except (requests.RequestException, KeyError, ValueError, TypeError) as exc:
        return {"status": "error", "city": key.title(), "message": f"Weather data unavailable: {exc}"}


## 2. Strict input validation and callbacks

Only a single supported city or simple weather question is accepted. This is an assignment-level allowlist, not a complete prompt-injection defense.

In [5]:
import logging
import re
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai import types

logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s", force=True)
logger = logging.getLogger("weather_callbacks")

CITY_RE = r"(?:los angeles|denver|washington|chantilly|manhattan)"
# Anchored patterns prevent appending arbitrary instructions after a city.
WEATHER_QUESTION = re.compile(
    rf"(?:what(?:'s| is) (?:the )?|check (?:the )?|show (?:me )?(?:the )?|"
    rf"give me (?:the )?)?"
    rf"(?:weather|temperature|humidity|precipitation|rain|forecast)"
    rf"(?:\s+(?:in|for|at))?\s+({CITY_RE})"
    rf"(?:\s+(?:today|right now|currently))?[?.!]*$",
    re.IGNORECASE,
)

BLOCKED_PATTERNS = [
    r"ignore.*instructions", r"disregard.*instructions",
    r"reveal.*system prompt", r"show.*system prompt",
    r"bypass.*rules", r"delete.*files", r"execute.*command",
    r"rm\s+-rf", r"<script", r"__import__", r"subprocess",
]

def parse_request(prompt: str):
    """Return (valid, city_or_none, message)."""
    if not isinstance(prompt, str) or not prompt.strip():
        return False, None, "Please enter a city or weather question."
    text = prompt.strip().lower()
    if len(text) > 500:
        return False, None, "Request too long (500 character maximum)."
    if any(re.search(p, text) for p in BLOCKED_PATTERNS):
        return False, None, "Request blocked: suspicious input."
    if not re.fullmatch(r"[a-z0-9\s,?.!'-]+", text):
        return False, None, "Request blocked: unsupported characters."
    if text.strip(" .!?") in {"hello", "hi", "hey", "help"}:
        return True, None, "Greeting accepted."
    city = text.strip(" .!?")
    if city in CITIES:
        return True, city, "Valid supported US city."
    match = WEATHER_QUESTION.fullmatch(text)
    if match:
        return True, match.group(1).lower(), "Valid supported US weather question."
    return False, None, ("Unsupported request or location. Choose Denver, Washington, "
                         "Chantilly, Manhattan, or Los Angeles (US only).")

def before_model_callback(callback_context: CallbackContext, llm_request: LlmRequest):
    """Log and validate the latest user prompt before the model is called."""
    messages = [part.text for content in (llm_request.contents or [])
                if content.role == "user" for part in (content.parts or []) if part.text]
    if not messages:
        return None
    prompt = messages[-1]
    logger.info("USER PROMPT: %s", prompt)
    valid, city, message = parse_request(prompt)
    if not valid:
        logger.warning("VALIDATION FAILED: %s", message)
        return LlmResponse(content=types.Content(role="model", parts=[types.Part(text=message)]))
    logger.info("VALIDATION PASSED: %s", message)
    return None

def after_model_callback(callback_context: CallbackContext, llm_response: LlmResponse):
    """Log simulated model output after execution."""
    if llm_response.content:
        for part in (llm_response.content.parts or []):
            if part.text:
                logger.info("MODEL RESPONSE: %s", part.text)
    return None

print("Callbacks loaded.")


Callbacks loaded.


## 3. Simulated ADK model and agent

In [6]:
from google.adk.models.base_llm import BaseLlm
from google.adk.agents import Agent

class SimulatedWeatherModel(BaseLlm):
    """Deterministic ADK model: no external model API calls."""

    async def generate_content_async(self, llm_request: LlmRequest, stream: bool = False):
        messages = [part.text for content in (llm_request.contents or [])
                    if content.role == "user" for part in (content.parts or []) if part.text]
        prompt = messages[-1] if messages else ""
        valid, city, message = parse_request(prompt)
        if not valid:
            reply = message  # Defense in depth if the callback is disabled.
        elif city is None:
            reply = ("Hello, I am your friendly weather neighborhood checker. "
                     "What can I help you with today?")
        else:
            weather = get_weather(city)
            if weather.get("status") != "success":
                reply = weather.get("message", "Weather data unavailable.")
            else:
                def fmt(value, suffix):
                    return "Not available" if value is None else f"{value}{suffix}"
                reply = (f"Weather for {weather['city']}:\n"
                         f"Temperature: {fmt(weather['temperature_f'], '°F')}\n"
                         f"Precipitation probability: {fmt(weather['precipitation_probability_percent'], '%')}\n"
                         f"Humidity: {fmt(weather['humidity_percent'], '%')}")
        yield LlmResponse(content=types.Content(role="model", parts=[types.Part(text=reply)]),
                          partial=False, turn_complete=True)

root_agent = Agent(
    name="neighborhood_weather_agent",
    model=SimulatedWeatherModel(model="simulated-weather"),
    description="NWS weather checker using a simulated ADK model and callbacks",
    instruction="Provide temperature, precipitation probability, and humidity for supported US cities.",
    before_model_callback=before_model_callback,
    after_model_callback=after_model_callback,
)
print("Simulated ADK agent ready; no API key required.")


Simulated ADK agent ready; no API key required.


6. Add VertexAI


In [7]:
import os
import google.auth
import vertexai

# Retrieve credentials and project information
credentials, project = google.auth.default()

# Configure environment variables for Vertex AI
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-02-29d1792d4bd6"
os.environ["VERTEX_AI_PROJECT"] = "qwiklabs-gcp-02-29d1792d4bd6"
os.environ["VERTEX_AI_LOCATION"] = "us-central1"

# Initialize the Vertex AI SDK
vertexai.init(
    project="qwiklabs-gcp-02-29d1792d4bd6",
    location="us-central1",
    credentials=credentials
)

print("Vertex AI initialized successfully!")

Vertex AI initialized successfully!


7. Add Google Search

In [8]:
from google.adk.tools import google_search

print("Google Search tool imported successfully!")

Google Search tool imported successfully!


8. Create Google Search Agent

In [9]:
from google.adk.models.lite_llm import LiteLlm

# Instantiate the Vertex AI model using the Google ADK wrapper
# This will leverage the Vertex AI project and credentials we configured earlier
MODEL = LiteLlm(model="gemini-2.5-flash")

print("Model 'MODEL' initialized successfully with gemini-2.5-flash!")

Model 'MODEL' initialized successfully with gemini-2.5-flash!


In [10]:
search_agent = Agent(
    name="search_agent",
    model=MODEL,
    description=(
        "Specialist for Google Search, research, "
        "current events, and general online information."
    ),
    instruction="""
    You are a Google Search specialist.

    Use the google_search tool to research user questions.

    Provide accurate, concise answers based on search results.

    Include sources when available.

    Do not handle NWS weather requests.
    Those should be handled by the weather_agent.
    """,
    tools=[google_search]
)

print("Google Search Agent created successfully!")

Google Search Agent created successfully!


In [11]:
#Recreate the sub-agents fresh to avoid any Pydantic parent-binding validation errors
weather_agent = Agent(
    name="weather_agent",
    model=MODEL,
    description=(
        "Specialist for current NWS weather information in "
        "Denver, Washington, Chantilly, Manhattan, and Los Angeles."
    ),
    instruction="""
    You are a friendly neighborhood weather specialist.

    You handle weather requests for these US cities:
    Denver, Washington, Chantilly, Manhattan, Los Angeles.

    Always call the get_weather tool for weather requests.

    Report only:
    1. Temperature in Fahrenheit
    2. Precipitation probability
    3. Humidity percentage

    Never invent weather information.

    If the user requests an unsupported city, explain
    which locations are supported.
    """,
    tools=[get_weather]
)

search_agent = Agent(
    name="search_agent",
    model=MODEL,
    description=(
        "Specialist for Google Search, research, "
        "current events, and general online information."
    ),
    instruction="""
    You are a Google Search specialist.

    Use the google_search tool to research user questions.

    Provide accurate, concise answers based on search results.

    Include sources when available.

    Do not handle NWS weather requests.
    Those should be handled by the weather_agent.
    """,
    tools=[google_search]
)

#Create the coordinating root agent with the fresh sub-agents
root_agent = Agent(
    name="root_agent",
    model=MODEL,
    description=(
        "Coordinating agent that delegates user requests "
        "to the Weather Agent or Google Search Agent."
    ),
    instruction="""
    You are the coordinating Root Agent.

    You manage two specialized sub-agents:

    1. weather_agent:
       Handles weather requests using the
       National Weather Service (NWS) API.

    2. search_agent:
       Handles Google searches, research,
       current events, and general information.

    Delegation Rules:

    - If the user asks about weather,
      delegate the request to weather_agent.

    - If the user asks for online research
      or general information,
      delegate the request to search_agent.

    - Do not answer requests yourself when
      a specialized agent can handle them.

    - If a request involves both weather and
      research, coordinate both agents.

    - Provide clear and accurate responses.
    """,
    sub_agents=[
        weather_agent,
        search_agent
    ]
)

print("Root Agent created successfully!")
print("Root Agent:", root_agent.name)
print(
    "Sub-Agents:",
    [agent.name for agent in root_agent.sub_agents]
)

Root Agent created successfully!
Root Agent: root_agent
Sub-Agents: ['weather_agent', 'search_agent']


In [12]:
import uuid
from google.adk.apps.app import App
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Configure App and Runner parameters with Context Caching enabled
APP_NAME = "weather_search_app"
USER_ID = "student"
SESSION_ID = str(uuid.uuid4())

# Create the App configuration with context caching enabled and pass the required root_agent
app = App(
    name=APP_NAME,
    root_agent=root_agent,
    context_cache_config=ContextCacheConfig()
)

# Initialize the session service (no apps argument needed in this version)
session_service = InMemorySessionService()
await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)

async def ask_agent(prompt):
    print(f"\nUSER: {prompt}")
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=types.Content(role="user", parts=[types.Part(text=prompt)]),
    ):
        if event.is_final_response() and event.content:
            for part in event.content.parts or []:
                if part.text:
                    print("AGENT:", part.text)


# Test routing to the Search Agent
await ask_agent("What is the capital of Utah?")

# Test routing to the Weather Agent
await ask_agent("What is the temperature in Denver?")


WARNING - App "weather_search_app" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after each transfer. Set context_cache_config on the app to give each agent its own cache.



USER: What is the capital of Utah?
00:17:29 - LiteLLM:INFO: utils.py:4706 - 
LiteLLM completion() model= gemini-2.5-flash; provider = vertex_ai


INFO - 
LiteLLM completion() model= gemini-2.5-flash; provider = vertex_ai


00:17:32 - LiteLLM:INFO: utils.py:4706 - 
LiteLLM completion() model= gemini-2.5-flash; provider = vertex_ai


INFO - 
LiteLLM completion() model= gemini-2.5-flash; provider = vertex_ai


AGENT: Salt Lake City is the capital and most populous city of the U.S. state of Utah. It has held this distinction since 1856. The city is also the county seat of Salt Lake County.

USER: What is the temperature in Denver?
00:17:34 - LiteLLM:INFO: utils.py:4706 - 
LiteLLM completion() model= gemini-2.5-flash; provider = vertex_ai


INFO - 
LiteLLM completion() model= gemini-2.5-flash; provider = vertex_ai


AGENT: The current temperature in Denver, Colorado is 52°F (11°C) and it feels like 51°F (10°C). The weather is clear, and the humidity is around 67%.The current temperature in Denver, Colorado is 52°F (11°C), and it feels like 51°F (10°C). The weather is clear with approximately 67% humidity.
